# Data Extraction: Commodities (10 Assets)


| Ticker | Commodity |
|--------|-----------|
| GC=F | Gold | 
| SI=F | Silver | 
| CL=F | Crude Oil (WTI) |
| NG=F | Natural Gas |
| HG=F | Copper |
| ZW=F | Wheat |
| ZC=F | Corn |
| ZS=F | Soybeans |
| PL=F | Platinum |
| KC=F | Coffee |

In [99]:
import yfinance as yf
import pandas as pd
import os
import time

In [100]:
START = "2011-01-01"
END   = "2025-12-31"

TICKERS = {
    "Gold":        "GC=F",
    "Silver":      "SI=F",
    "Crude_Oil":   "CL=F",
    "Natural_Gas": "NG=F",
    "Copper":      "HG=F",
    "Wheat":       "ZW=F",
    "Corn":        "ZC=F",
    "Soybeans":    "ZS=F",
    "Platinum":    "PL=F",
    "Coffee":      "KC=F",
}


In [ ]:
raw = {}
for name, ticker in TICKERS.items():
    print(f"Downloading {name} ({ticker})...")
    for attempt in range(1, 6):
        try:
            df = yf.download(
                ticker,
                start=START,
                end=END,
                auto_adjust=True,
                progress=False,
                threads=False,
            )
            if not df.empty:
                break
        except Exception as e:
            df = pd.DataFrame()
            print(f"  Error: {e}")
        wait = attempt * 30
        print(f"  Retrying in {wait}s (attempt {attempt}/5)...")
        time.sleep(wait)

    if df.empty:
        print(f"  FAILED: {name}")
        continue

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df.index = pd.to_datetime(df.index)
    raw[name] = df
    print(f"  {len(df)} rows  |  {df.index[0].date()} -> {df.index[-1].date()}")
    time.sleep(2)

  3770 rows  |  2011-01-03 -> 2025-12-30
  3770 rows  |  2011-01-03 -> 2025-12-30


KeyboardInterrupt: 

In [ ]:
for name, df in raw.items():
    print(f"\n--- {name} ---")
    print(df.shape)
    print(df.isna().sum())
    display(df.tail())

In [ ]:
for name, df in raw.items():
    print(f"\n--- {name} ---")
    display(df["Close"].describe())

In [ ]:
os.makedirs("../data/commodities", exist_ok=True)

for name, df in raw.items():
    out = f"../data/commodities/raw_{name.lower()}.csv"
    df.to_csv(out)
    print(f"Saved {out}")